In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict
load_dotenv()

model= ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


In [3]:
class BlogState(TypedDict):

    title:str
    outline:str
    content:str

In [5]:
def create_outline(state:BlogState)->BlogState:
    #fetch title
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state

In [10]:
def create_blog(state: BlogState):
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline\n{outline}'

    content = model.invoke(prompt).content

    state['content']  = content

    return state

In [12]:
graph = StateGraph(BlogState)
#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow=graph.compile()

In [13]:
initial_state = {'title':'story behind eternal sunshine of the spotless mind'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'story behind eternal sunshine of the spotless mind', 'outline': '**Blog Title:** *The Story Behind “Eternal\u202fSunshine of the Spotless Mind” – From Conception to Cultural Phenomenon*  \n\n---\n\n## 1. Introduction  \n- **Hook:** A striking opening line (e.g., “What if you could erase the pain of a breakup… only to discover that love is a memory you can’t delete?”).  \n- **Brief Synopsis:** One‑sentence summary of the film’s premise.  \n- **Purpose of the Post:** Explain why the film’s back‑story matters—its influence on cinema, storytelling, and our understanding of memory.  \n- **Thesis Statement:** The blog will unpack the film’s origins, creative forces, production challenges, thematic depth, and lasting impact.\n\n---\n\n## 2. The Seed of an Idea  \n### 2.1. Charlie Kaufman’s Original Concept  \n- Early 1990s: Kaufman’s work as a television writer (e.g., *The Dana Carvey Show*).  \n- The “memory‑erasing” pitch: a short story titled **“Eternal Sunshine of the Spotless 

In [16]:
print(final_state['content'])

# The Story Behind “Eternal Sunshine of the Spotless Mind” – From Conception to Cultural Phenomenon  

---

## 1. Introduction  

**Hook:** *What if you could erase the pain of a breakup… only to discover that love is a memory you can’t delete?*  

**Brief Synopsis:** *Eternal Sunshine of the Spotless Mind* follows Joel (Jim Carrey) and Clementine (Kate Winslet) as they undergo a clandestine procedure to wipe each other from their minds—only to realize that some memories, however painful, are worth keeping.  

**Purpose of the Post:** Understanding the film’s back‑story reveals why it reshaped modern cinema, opened new avenues for storytelling about memory, and continues to influence how we think about love, loss, and the mind.  

**Thesis Statement:** This post unpacks the film’s origins, the creative forces that birthed it, the production hurdles it overcame, its layered themes, and the lasting cultural imprint it has left on both the industry and audiences worldwide.  

---  

## 2.